# Part 4 - Video Nutrition Label (KuaiRec)
**Author:** Imad eddine Djehlane

**What this notebook does.** Food has an ingredient label. Short video has nothing. This notebook builds a
"video nutrition label" for every user in KuaiRec: how many videos they watch per day, what the time is
made of (categories = "ingredients"), how much is replays ("added sugar"), night viewing and the longest
binge. It then writes a short plain-language summary with a language model and builds a clickable
prototype (`label_prototype.html`) that opens in any browser.

**How to use (Google Colab).** colab.research.google.com -> File -> Upload notebook. Run the cells from top
to bottom (Shift+Enter). The only settings you may change are marked `<<< EDIT`. Without a Gemini key the
notebook still runs: the summary is then a rule-based text, and it says so.

**Data.** KuaiRec (Gao et al., 2022, CIKM '22). We use `big_matrix.csv` (natural viewing, with timestamps
and play time) and `kuairec_caption_category.csv` (one category per video). `small_matrix.csv` is used
only to rebuild the same 100-user sample as Roles 2 and 3 (seed 42).

## Step 0 - Settings

In [ ]:
!pip -q install pandas numpy matplotlib google-genai

In [ ]:
import os, re, json, glob, time, getpass, html
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

SEED = 42                 # same as Roles 2 and 3
N_USERS = 100             # same shared sample as Roles 2 and 3
SESSION_GAP_MIN = 30      # a pause of 30+ minutes starts a new "sitting" (common web-analytics convention)
NIGHT_HOURS = range(0, 6) # 00:00-05:59, platform clock in the data
SKIP_RATIO = 0.25         # "quick swipe" = less than a quarter of the video watched
TOP_INGREDIENTS = 5       # categories shown by name on the label, the rest is "Other"

USE_GEMINI = False        # <<< EDIT: True = write the summaries with Gemini (free key from aistudio.google.com/apikey)
GEMINI_MODELS = ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-flash-latest"]
OUT = Path(os.environ.get("LABEL_OUT", "label_outputs"))   # where every output file goes
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(exist_ok=True)
print("Outputs go to:", OUT.resolve())

## Step 1 - Find or download the data
If the download fails, download `KuaiRec.zip` by hand from https://zenodo.org/records/18164998, upload it in
the Colab file panel (left side) and run this cell again.

In [ ]:
SEARCH_ROOTS = [os.environ.get("KUAIREC_DIR", ""), ".", "data/raw", "../data/raw", "../../data/raw"]

def find_file(name):
    for root in SEARCH_ROOTS:
        if root and os.path.isdir(root):
            hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
            if hits:
                return hits[0]
    return None

if not find_file("big_matrix.csv"):
    if not os.path.exists("KuaiRec.zip"):
        os.system("wget -q -O KuaiRec.zip 'https://zenodo.org/records/18164998/files/KuaiRec.zip?download=1'")
    os.system("unzip -q -o KuaiRec.zip")
BIG, SMALL, CAP = (find_file(n) for n in ["big_matrix.csv", "small_matrix.csv", "kuairec_caption_category.csv"])
print(BIG, SMALL, CAP, sep="\n")

## Step 2 - Load viewing data and categories
One row of `big_matrix` = one user watched one video once. `play_duration` and `video_duration` are in
milliseconds. `time` is the platform's clock, `date` is the logging day (28 days between July and September 2020).

In [ ]:
big = pd.read_csv(BIG, usecols=["user_id", "video_id", "play_duration", "video_duration", "time", "date", "timestamp"],
                  dtype={"user_id": "int32", "video_id": "int32", "play_duration": "int64",
                         "video_duration": "int64", "timestamp": "float64", "time": "string", "date": "string"})

# Categories: first-level category, translated (same mapping as src/categories.py in the team repo).
ENGLISH = {1: "Dance", 2: "Music", 3: "Gaming", 4: "Beauty & makeup", 5: "Fashion", 6: "Celebrities",
           7: "Sports", 8: "Good looks", 9: "Comedy", 10: "Travel", 11: "Lifestyle", 12: "Food",
           13: "Rural life", 14: "Education", 15: "Arts", 16: "Health", 17: "Pets", 18: "Cars",
           19: "Relationships", 20: "Anime & manga", 21: "Culture & history", 22: "Finance", 23: "Politics",
           24: "Astrology", 25: "Parenting", 26: "Photography", 27: "Tech & gadgets", 28: "Local news",
           29: "Science & law", 31: "Fitness", 32: "Short drama", 33: "Selfies", 34: "Other topics",
           35: "Military", 36: "Home & real estate", 37: "Oddities", 38: "Books", 39: "Film & TV",
           -124: "Unknown"}
cap = pd.read_csv(CAP, usecols=["video_id", "first_level_category_id"], dtype=str, engine="python")
cap = cap[cap["video_id"].str.fullmatch(r"\d+", na=False)]          # a few captions break rows in two
cats = pd.Series(cap["first_level_category_id"].astype(float).fillna(-124).astype(int).map(ENGLISH).values,
                 index=cap["video_id"].astype(int).values)
big["category"] = big["video_id"].map(cats).fillna("Unknown")
print(f"{len(big):,} views | {big.user_id.nunique():,} users | {big.video_id.nunique():,} videos | "
      f"{big.category.nunique()} categories")

## Step 3 - Per-view features
- **seconds watched** = play time. A video replayed on loop can count more seconds than its length.
- **replay seconds** ("added sugar") = seconds watched beyond one full play of the video.
- **quick swipe** = less than a quarter of the video watched.
- **night** = watched between 00:00 and 05:59 (platform clock).
- **sitting** (session) = a run of views with no pause of 30 minutes or more.

In [ ]:
big["sec"] = big["play_duration"] / 1000
big["replay_sec"] = (big["play_duration"] - big["video_duration"]).clip(lower=0) / 1000
big["skip"] = big["play_duration"] < SKIP_RATIO * big["video_duration"]
big["hour"] = big["time"].str.slice(11, 13).astype(int)
big["day"] = big["date"]   # the 28 official logging days (0.1% of rows have a "time" outside them)
big["night"] = big["hour"].isin(list(NIGHT_HOURS))
big = big.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
gap = big.groupby("user_id")["timestamp"].diff()
big["sitting"] = (gap.isna() | (gap > SESSION_GAP_MIN * 60)).cumsum()

# How much of the real viewing does KuaiRec log? Within a sitting, compare the logged play time with the
# time between the first and the last logged view. KuaiRec only contains 10,728 videos, so most of what a
# user watched in a sitting is not in the file. "Time in the app" below is therefore estimated from the
# sitting spans (first to last logged view). We tested it and REJECTED it for the label: a pause of up
# to 29 minutes counts as "in the app", so it gives a median of almost 4 hours a day and some 18-hour
# sittings. The label therefore counts videos, and shows logged minutes as a secondary line.
s = big.groupby("sitting").agg(t0=("timestamp", "min"), t1=("timestamp", "max"),
                               last=("sec", "last"), played=("sec", "sum"))
s["span_min"] = ((s["t1"] - s["t0"]) + s["last"]) / 60
COVERAGE = float(s["played"].sum() / (s["span_min"].sum() * 60))
print(f"Logged play time = {COVERAGE:.1%} of the time between first and last view of each sitting")

## Step 4 - One label per user (all 7,176 users)

In [ ]:
def ingredients(g):
    by_cat = g.groupby("category")["sec"].sum().sort_values(ascending=False)
    share = by_cat / by_cat.sum()
    n80 = int((share.cumsum() < 0.8).sum() + 1)          # categories needed to reach 80% of the time
    return share, n80

rows, compositions = [], {}
for uid, g in big.groupby("user_id", sort=True):
    share, n80 = ingredients(g)
    days = g["day"].nunique()
    sit = g.groupby("sitting").size()
    longest = int(sit.max())
    longest_day = g.loc[g["sitting"] == sit.idxmax(), "day"].iloc[0]
    tot = g["sec"].sum()
    spans = s.loc[sit.index, "span_min"]
    rows.append({"user_id": int(uid), "days": days, "videos_per_day": len(g) / days,
                 "minutes_per_day": tot / 60 / days, "app_minutes_per_day": float(spans.sum()) / days,
                 "longest_sitting_minutes": float(spans.loc[sit.idxmax()]), "top_category": share.index[0],
                 "top_share": float(share.iloc[0]), "cats_for_80": n80, "cats_total": int(len(share)),
                 "replay_share": float(g["replay_sec"].sum() / tot) if tot else 0.0,
                 "night_share": float(g.loc[g["night"], "sec"].sum() / tot) if tot else 0.0,
                 "skip_share": float(g["skip"].mean()), "longest_sitting": longest,
                 "longest_sitting_day": longest_day, "sittings_per_day": len(sit) / days})
    compositions[int(uid)] = share
labels = pd.DataFrame(rows).set_index("user_id")

# IMPORTANT data trap. The 1,411 small-matrix users have an INCOMPLETE history in big_matrix: KuaiRec moved
# their views of the 3,327 small-matrix videos out of big_matrix (that is the test set). They look like
# very light viewers (median ~15 videos a day vs ~82 for everyone else). So the reference population for
# every comparison is the 5,765 users with a complete history ("complete" below).
small_users = set(pd.read_csv(SMALL, usecols=["user_id"])["user_id"].unique())
labels["complete"] = ~labels.index.isin(small_users)
ref = labels[labels["complete"]]
print(labels.groupby("complete")["videos_per_day"].median().rename("median videos/day"))

# Where does each user sit compared with the complete users? 0.82 = "higher than 82% of users".
METRICS = ["videos_per_day", "minutes_per_day", "top_share", "cats_for_80", "replay_share",
           "night_share", "skip_share", "longest_sitting"]
for m in METRICS:
    sv = np.sort(ref[m].values)
    labels[m + "_pct"] = (np.searchsorted(sv, labels[m].values, "left") +
                          np.searchsorted(sv, labels[m].values, "right")) / 2 / len(sv)
ref = labels[labels["complete"]]
labels.round(4).to_csv(OUT / "labels_all_users.csv")
labels[METRICS].describe(percentiles=[.1, .5, .9]).round(3)

## Step 5 - Population findings (for the report)

In [ ]:
# All population numbers use the complete users only (see the data trap in Step 4).
refv = big[big["user_id"].isin(ref.index)]
pop = {"n_users": int(len(ref)), "n_users_all": int(len(labels)), "n_views": int(len(refv)),
       "coverage": COVERAGE, "n_days_in_data": int(big["day"].nunique()),
       "small_users_median_videos_per_day": float(labels.loc[~labels["complete"], "videos_per_day"].median()),
       "small_users_median_longest_sitting": float(labels.loc[~labels["complete"], "longest_sitting"].median())}
for m in METRICS:
    pop[m] = {q: float(ref[m].quantile(v)) for q, v in [("p10", .1), ("p50", .5), ("p90", .9)]}
pop["replay_share_all_time"] = float(refv["replay_sec"].sum() / refv["sec"].sum())
pop["night_share_all_time"] = float(refv.loc[refv["night"], "sec"].sum() / refv["sec"].sum())

# Do heavier users have a narrower diet? Spearman correlation across users.
pop["spearman"] = {m: float(ref["videos_per_day"].corr(ref[m], method="spearman"))
                   for m in ["top_share", "cats_for_80", "replay_share", "night_share", "skip_share"]}
ref = ref.assign(volume_band=pd.qcut(ref["videos_per_day"], 5, labels=["lightest 20%", "2", "3", "4", "heaviest 20%"]))
bands = ref.groupby("volume_band", observed=True).agg(
    videos_per_day=("videos_per_day", "median"), top_share=("top_share", "median"),
    cats_for_80=("cats_for_80", "median"), replay_share=("replay_share", "median"),
    night_share=("night_share", "median"))
bands.round(3).to_csv(OUT / "volume_bands.csv")

# Ingredient table: what each category is "made of".
cat_tab = refv.groupby("category").agg(time_share=("sec", "sum"), replay_sec=("replay_sec", "sum"),
                                      skip_rate=("skip", "mean"), views=("sec", "size"))
cat_tab["replay_share"] = cat_tab["replay_sec"] / cat_tab["time_share"]
cat_tab["time_share"] = cat_tab["time_share"] / cat_tab["time_share"].sum()
cat_tab = cat_tab.drop(columns="replay_sec").sort_values("time_share", ascending=False)
cat_tab.round(4).to_csv(OUT / "category_ingredients.csv")
hours = (refv.groupby("hour")["sec"].sum() / refv["sec"].sum())
pop["hour_share"] = {int(h): float(v) for h, v in hours.items()}
pop["peak_hour"], pop["low_hour"] = int(hours.idxmax()), int(hours.idxmin())
pop["rejected_app_minutes_per_day_p50"] = float(ref["app_minutes_per_day"].median())
pop["rejected_longest_sitting_minutes_max"] = float(ref["longest_sitting_minutes"].max())
del refv
print(json.dumps(pop["spearman"], indent=1))
bands.round(3)

In [ ]:
ORANGE, INK, MUTED, GRID = "#e85d26", "#1f1f1c", "#77776f", "#e4e2dc"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.titleweight": "bold", "axes.titlesize": 13})

# Figure 1: how many videos per day do users watch?
fig, ax = plt.subplots(figsize=(8, 4))
v = ref["videos_per_day"].clip(upper=300)
ax.hist(v, bins=60, color=ORANGE, edgecolor="white", linewidth=0.5)
med = ref["videos_per_day"].median()
ax.axvline(med, color=INK, lw=1.5, ls="--")
ax.text(med + 4, ax.get_ylim()[1] * 0.9, f"median user: {med:.0f} videos a day", color=INK)
ax.set_xlabel("Logged videos per active day (5,765 users with a complete history; above 300 shown at 300)")
ax.set_ylabel("Number of users")
ax.set_title("Most users watch dozens of logged videos a day, a few watch hundreds")
fig.tight_layout(); fig.savefig(OUT / "figures/fig1_videos_per_day.png", dpi=200); plt.show()

# Figure 2: replays ("added sugar") by category, top 15 categories by time.
top15 = cat_tab.head(15).sort_values("replay_share")
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.barh(top15.index, top15["replay_share"] * 100, color=ORANGE, height=0.6)
ax.axvline(pop["replay_share_all_time"] * 100, color=INK, lw=1.5, ls="--")
ax.text(pop["replay_share_all_time"] * 100 + 0.3, -0.9, f"all videos: {pop['replay_share_all_time']:.0%}", color=INK)
ax.set_xlabel("Share of watch time that is replay (% of seconds beyond one full play)")
ax.set_title("'Added sugar': how much of each category's time is replays")
ax.grid(axis="x", color=GRID); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(OUT / "figures/fig2_replay_by_category.png", dpi=200); plt.show()

# Figure 3: heavier users vs. how narrow their diet is.
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(bands))
ax.bar(x, bands["top_share"] * 100, color=ORANGE, width=0.6)
for i, (vpd, ts) in enumerate(zip(bands["videos_per_day"], bands["top_share"])):
    ax.text(i, ts * 100 + 0.6, f"{ts:.0%}", ha="center", color=INK)
ax.set_xticks(x, [f"{b}\n(~{vpd:.0f} videos/day)" for b, vpd in zip(bands.index, bands["videos_per_day"])])
ax.set_ylabel("Median share of time in\nthe user's top category (%)")
ax.set_title("Heavier users do not have a narrower diet")
ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(OUT / "figures/fig3_volume_vs_concentration.png", dpi=200); plt.show()

# Figure 4: time of day.
fig, ax = plt.subplots(figsize=(8, 3.6))
colors = [ORANGE if h in NIGHT_HOURS else "#c9c6bd" for h in hours.index]
ax.bar(hours.index, hours.values * 100, color=colors, width=0.75)
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("Hour of day (platform clock in the data)")
ax.set_ylabel("% of all watch time")
ax.set_title("When the logged viewing happens (orange = our 'night' window, 00:00-05:59)")
ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(OUT / "figures/fig4_hour_of_day.png", dpi=200); plt.show()

## Step 6 - The shared 100-user sample (identical to Roles 2 and 3)
Same code as the Role 3 notebook: users drawn from `small_matrix` with seed 42, kept only if they have at
least one liked video (watch_ratio > 2) with a caption entry. We also pick three contrasting "demo users"
by a fixed rule, so nobody can say we cherry-picked.

In [ ]:
small = pd.read_csv(SMALL, usecols=["user_id", "video_id", "watch_ratio"])   # only to rebuild the shared sample
users = sorted(small.user_id.unique())
sample = np.random.RandomState(SEED).choice(users, N_USERS, replace=False)
has_like = set(small[small.video_id.isin(cats.index) & (small.watch_ratio > 2)].user_id.unique())
shared = [int(u) for u in sample if int(u) in has_like]
del small
print(len(shared), "shared users")
print("Shared users, median videos/day:", round(labels.loc[shared, "videos_per_day"].median(), 1),
      "| complete users:", round(ref["videos_per_day"].median(), 1))
pop["shared_median_videos_per_day"] = float(labels.loc[shared, "videos_per_day"].median())
pop["shared_max_videos_per_day"] = float(labels.loc[shared, "videos_per_day"].max())
json.dump(pop, open(OUT / "population_summary.json", "w"), indent=2)

# Demo users are chosen from the complete users, by a fixed rule, among users with a typical amount of viewing
# (between the 40th and 60th percentile of videos per day), so the three labels are comparable.
# Why not from the shared 100? They are all light viewers in big_matrix (see the report): KuaiRec removed
# their small-matrix videos from their big-matrix history, so their labels miss part of their viewing.
typical = ref[ref["videos_per_day"].between(ref["videos_per_day"].quantile(.4),
                                            ref["videos_per_day"].quantile(.6))]
narrow = int(typical["top_share"].idxmax())                                   # narrowest diet
broad = int(typical["top_share"].idxmin())                                    # broadest diet
heavy = int((ref["videos_per_day"] - ref["videos_per_day"].quantile(.95)).abs().idxmin())  # p95 viewer
DEMO = {narrow: "The narrow diet", broad: "The broad diet", heavy: "The heavy viewer"}
print(DEMO)

## Step 7 - The label as data, and the AI summary
The language model only ever sees the numbers on the label (no raw history), and is told to stay neutral:
no diagnosis, no moralising, only numbers from the input.

In [ ]:
def pct_text(p):
    return f"more than {p:.0%} of users" if p >= 0.5 else f"less than {1 - p:.0%} of users"

shared_set = set(shared)

def label_dict(uid):
    r = labels.loc[uid]
    share = compositions[uid]
    top = share.head(TOP_INGREDIENTS)
    ing = [{"name": c, "share": round(float(v), 4)} for c, v in top.items()]
    rest = 1 - float(top.sum())
    if rest > 0.0005:
        ing.append({"name": f"Other ({len(share) - len(top)} categories)", "share": round(rest, 4)})
    d = {"user_id": int(uid), "persona": DEMO.get(uid, ""), "days": int(r.days),
         "videos_per_day": round(float(r.videos_per_day), 1), "minutes_per_day": round(float(r.minutes_per_day), 1),
         "shared_sample": int(uid) in shared_set,
         "top_category": r.top_category, "top_share": round(float(r.top_share), 3),
         "cats_for_80": int(r.cats_for_80), "cats_total": int(r.cats_total),
         "replay_share": round(float(r.replay_share), 3), "night_share": round(float(r.night_share), 3),
         "skip_share": round(float(r.skip_share), 3), "longest_sitting": int(r.longest_sitting),
         "longest_sitting_day": r.longest_sitting_day, "sittings_per_day": round(float(r.sittings_per_day), 1),
         "ingredients": ing}
    d["pct"] = {m: round(float(r[m + "_pct"]), 3) for m in METRICS}
    return d

PROMPT = """You write the summary on a "video nutrition label" for a short-video app user.
Facts (JSON, all numbers are averages per active day over {days} days of logged viewing):
{facts}
Population medians for comparison: {medians}

Write exactly 3 short sentences in plain English, second person ("you").
1. What the viewing is mostly made of (top categories with their % of time).
2. The one thing that stands out most compared with other users (use the percentile you are given).
3. One gentle, optional suggestion, phrased as a choice, not an order.
Rules: use only numbers that appear in the facts; no medical or psychological claims; no moralising;
no word "addiction"; under 75 words. Reply with the 3 sentences only."""

def prompt_for(d):
    facts = {k: d[k] for k in ["videos_per_day", "ingredients", "top_category", "top_share", "cats_for_80",
                               "replay_share", "night_share", "skip_share", "longest_sitting", "pct"]}
    med = {m: round(pop[m]["p50"], 3) for m in METRICS}
    return PROMPT.format(days=d["days"], facts=json.dumps(facts), medians=json.dumps(med))

def rule_summary(d):
    a, b = d["ingredients"][0], d["ingredients"][1]
    s1 = (f"About {a['share']:.0%} of your viewing time goes to {a['name']} and {b['share']:.0%} to {b['name']}, "
          f"across {d['videos_per_day']:.0f} videos a day.")
    standouts = {"videos_per_day": f"you watch more videos a day than {d['pct']['videos_per_day']:.0%} of users",
                 "top_share": f"your top category takes a bigger share of your time than for {d['pct']['top_share']:.0%} of users",
                 "replay_share": f"{d['replay_share']:.0%} of your time is replays, more than for {d['pct']['replay_share']:.0%} of users",
                 "night_share": f"{d['night_share']:.0%} of your time is between midnight and 6 am, more than for {d['pct']['night_share']:.0%} of users",
                 "longest_sitting": f"your longest sitting was {d['longest_sitting']} videos without a 30-minute break"}
    key = max(standouts, key=lambda k: d["pct"][k])
    s2 = "What stands out: " + standouts[key] + "."
    if d["pct"]["top_share"] < 0.25:
        s2 = f"What stands out: your mix is broad, with 80% of your time spread over {d['cats_for_80']} categories."
        return " ".join([s1, s2, "If you want, you could check this label now and then to see if it stays that way."])
    s3 = {"videos_per_day": "If you want, you could set a daily video budget and see how it feels.",
          "top_share": "If you want, you could follow one new topic this week and see if your feed widens.",
          "replay_share": "If you want, you could turn off autoplay loops for a week.",
          "night_share": "If you want, you could set a quiet-hours reminder after midnight.",
          "longest_sitting": "If you want, you could set a gentle break reminder every 30 minutes."}[key]
    return " ".join([s1, s2, s3])

SUMMARY_FILE = OUT / "ai_summaries.json"
summaries = json.load(open(SUMMARY_FILE, encoding="utf-8")) if SUMMARY_FILE.exists() else {}

if USE_GEMINI:
    from google import genai   # Colab: !pip -q install google-genai
    client = genai.Client(api_key=getpass.getpass("Paste your FREE Gemini API key and press Enter: "))
    for uid in list(DEMO) + [u for u in shared if u not in DEMO]:
        if str(uid) in summaries and summaries[str(uid)]["source"].startswith("Gemini"):
            continue
        for model in GEMINI_MODELS:
            try:
                r = client.models.generate_content(model=model, contents=prompt_for(label_dict(uid)))
                summaries[str(uid)] = {"text": (r.text or "").strip(), "source": f"Gemini ({model})"}
                time.sleep(4)          # stay under the free rate limit
                break
            except Exception as e:
                print(model, "failed:", str(e)[:120])
        json.dump(summaries, open(SUMMARY_FILE, "w", encoding="utf-8"), indent=1, ensure_ascii=False)

print("Example prompt:\n", prompt_for(label_dict(narrow)))

## Step 8 - Build the clickable prototype (`label_prototype.html`)
One self-contained HTML file: open it in any browser, no installation, no internet needed (fonts fall back).

In [ ]:
# The prototype's HTML template (you don't need to read or change this).
TEMPLATE = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Video Nutrition Label · prototype</title>
<link rel="preconnect" href="https://fonts.googleapis.com" />
<link href="https://fonts.googleapis.com/css2?family=Barlow:wght@400;500;700;800;900&family=IBM+Plex+Mono:wght@400;500&display=swap" rel="stylesheet" />
<style>
  :root {
    --bg: #111111; --bg2: #1a1a18; --fg: #f0ece5; --fg2: #a8a59c; --fg3: #6b6b63;
    --accent: #e85d26; --border: #2e2e2a;
    --display: "Barlow", "Helvetica Neue", Arial, sans-serif;
    --mono: "IBM Plex Mono", ui-monospace, Menlo, Consolas, monospace;
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--bg); color: var(--fg); font-family: var(--display); font-size: 16px; line-height: 1.45; }
  .wrap { max-width: 1240px; margin: 0 auto; padding: 32px 28px 60px; }
  .mono { font-family: var(--mono); font-size: 12px; letter-spacing: .06em; text-transform: uppercase; color: var(--fg2); }
  header.top { display: flex; justify-content: space-between; align-items: baseline; border-bottom: 1px solid var(--border); padding-bottom: 14px; flex-wrap: wrap; gap: 8px; }
  h1 { font-weight: 900; font-size: 52px; line-height: .95; letter-spacing: -.03em; margin: 28px 0 8px; }
  h1 span { color: var(--accent); }
  .lead { color: var(--fg2); font-size: 19px; max-width: 760px; margin: 0 0 26px; }
  h2 { font-weight: 800; font-size: 28px; letter-spacing: -.01em; margin: 56px 0 6px; }
  .sub { color: var(--fg2); margin: 0 0 22px; max-width: 820px; }

  .picker { display: flex; gap: 10px; flex-wrap: wrap; align-items: center; margin-bottom: 28px; }
  .picker button { font-family: var(--display); font-weight: 700; font-size: 15px; background: transparent; color: var(--fg);
    border: 1px solid var(--fg3); padding: 9px 16px; cursor: pointer; }
  .picker button:hover { border-color: var(--fg); }
  .picker button.on { background: var(--accent); border-color: var(--accent); color: #111; }
  .picker select { font-family: var(--mono); font-size: 13px; background: var(--bg2); color: var(--fg); border: 1px solid var(--fg3); padding: 9px 10px; }
  .note { font-size: 13px; color: var(--fg2); border-left: 3px solid var(--accent); padding: 6px 12px; margin: -12px 0 24px; display: none; max-width: 820px; }

  .main { display: grid; grid-template-columns: 300px 400px 1fr; gap: 32px; align-items: start; }
  @media (max-width: 1100px) { .main { grid-template-columns: 1fr 1fr; } .main .ai { grid-column: 1 / -1; } }
  @media (max-width: 720px) { .main { grid-template-columns: 1fr; } h1 { font-size: 38px; } }

  /* bottle */
  .bottle svg { width: 100%; max-width: 230px; display: block; margin: 0 auto; }
  .legend { margin-top: 14px; }
  .legend div { display: grid; grid-template-columns: 14px 1fr auto; gap: 10px; align-items: center; padding: 4px 0; font-size: 15px; border-bottom: 1px solid var(--border); }
  .legend i { width: 14px; height: 14px; display: block; border-radius: 2px; }
  .legend b { font-family: var(--mono); font-weight: 500; font-size: 13px; color: var(--fg); }
  .layer:hover { opacity: .8; }
  #tip { position: fixed; pointer-events: none; background: var(--fg); color: #111; font-size: 13px; padding: 6px 9px; display: none; font-weight: 600; z-index: 10; }

  /* the label: deliberately the classic black-on-white food label */
  .label { background: #fff; color: #111; border: 2px solid #111; padding: 10px 12px 12px; font-family: Arial, Helvetica, sans-serif; }
  .label h3 { font-family: "Arial Black", Arial, sans-serif; font-weight: 900; font-size: 34px; line-height: 1; margin: 0 0 4px; letter-spacing: -.02em; }
  .label .srv { font-size: 14px; border-bottom: 10px solid #111; padding-bottom: 5px; }
  .label .row { display: grid; grid-template-columns: 1fr auto; gap: 8px; padding: 4px 0; border-top: 1px solid #111; font-size: 14px; align-items: baseline; }
  .label .row .cmp { grid-column: 1 / -1; font-size: 11.5px; color: #555; margin-top: -2px; }
  .label .big { font-size: 15px; font-weight: 800; border-top: none; }
  .label .big .v { font-size: 30px; font-weight: 900; }
  .label .thick { border-top: 5px solid #111; }
  .label .head { font-size: 11.5px; font-weight: 700; text-align: right; border-top: 1px solid #111; padding: 3px 0; }
  .label .ing { padding-left: 14px; }
  .label .ing.first { padding-left: 0; font-weight: 800; }
  .label .foot { border-top: 10px solid #111; margin-top: 4px; padding-top: 6px; font-size: 11px; color: #333; line-height: 1.35; }
  .bar { display: inline-block; height: 7px; width: 70px; background: #e3e1dc; position: relative; vertical-align: middle; margin-left: 6px; }
  .bar i { position: absolute; top: -3px; width: 3px; height: 13px; background: #111; }
  .bar.hi i { background: var(--accent); }

  /* AI summary */
  .ai .card { border: 1px solid var(--border); background: var(--bg2); padding: 22px 22px 18px; }
  .ai .q { font-size: 21px; line-height: 1.45; color: var(--fg); }
  .ai .src { margin-top: 16px; }
  .ai .why { margin-top: 18px; font-size: 14px; color: var(--fg2); }
  .ai .why li { margin-bottom: 4px; }

  /* three moments */
  .moments { display: grid; grid-template-columns: repeat(3, 1fr); gap: 28px; }
  @media (max-width: 900px) { .moments { grid-template-columns: 1fr; } }
  .moment .mono { color: var(--accent); }
  .moment h4 { font-size: 21px; font-weight: 800; margin: 4px 0 12px; }
  .phone { width: 230px; height: 430px; border: 2px solid var(--fg3); border-radius: 30px; padding: 12px; margin: 0 auto 12px; background: #000; }
  .screen { position: relative; width: 100%; height: 100%; border-radius: 20px; overflow: hidden;
    background: linear-gradient(160deg, #3b2a22 0%, #1d1d1b 55%, #23302f 100%); }
  .screen .play { position: absolute; left: 50%; top: 45%; transform: translate(-50%, -50%); width: 0; height: 0;
    border-left: 26px solid rgba(240,236,229,.35); border-top: 16px solid transparent; border-bottom: 16px solid transparent; }
  .chip { position: absolute; top: 12px; left: 10px; right: 10px; background: rgba(17,17,17,.82); border: 1px solid var(--accent);
    color: var(--fg); font-size: 12px; padding: 7px 9px; border-radius: 10px; }
  .chip b { color: var(--accent); }
  .sheet { position: absolute; left: 8px; right: 8px; bottom: 8px; background: var(--fg); color: #111; border-radius: 14px; padding: 12px; font-size: 12.5px; }
  .sheet h5 { margin: 0 0 6px; font-size: 15px; }
  .stack { display: flex; height: 14px; margin: 8px 0 4px; gap: 2px; }
  .stack span { display: block; height: 100%; }
  .btns { display: flex; gap: 6px; margin-top: 10px; }
  .btns div { flex: 1; text-align: center; padding: 7px 0; border-radius: 8px; font-weight: 700; border: 1.5px solid #111; }
  .btns div.p { background: #111; color: var(--fg); }
  .mini { position: absolute; inset: 8px; background: var(--fg); color: #111; border-radius: 14px; padding: 12px; font-size: 12px; overflow: hidden; }
  .mini h5 { font-family: "Arial Black", Arial, sans-serif; font-size: 17px; margin: 0 0 4px; border-bottom: 5px solid #111; padding-bottom: 3px; }
  .mini p { margin: 6px 0 0; line-height: 1.4; }
  .moment p.cap { font-size: 14px; color: var(--fg2); margin: 0; }

  table.who { width: 100%; border-collapse: collapse; font-size: 15px; }
  table.who th, table.who td { text-align: left; padding: 10px 12px; border-top: 1px solid var(--border); vertical-align: top; }
  table.who th { font-family: var(--mono); font-size: 12px; font-weight: 500; text-transform: uppercase; letter-spacing: .06em; color: var(--fg2); }
  table.who td:first-child { font-weight: 700; width: 22%; }
  .yes { color: var(--fg); } .no { color: var(--fg3); }
  .notes { margin-top: 56px; border-top: 1px solid var(--border); padding-top: 18px; font-size: 13.5px; color: var(--fg2); columns: 2; column-gap: 40px; }
  @media (max-width: 800px) { .notes { columns: 1; } }
  .notes p { margin: 0 0 10px; break-inside: avoid; }
  .notes b { color: var(--fg); }
</style>
</head>
<body>
<div id="tip"></div>
<div class="wrap">
  <header class="top">
    <span class="mono">Part 4 · Video nutrition label · prototype</span>
    <span class="mono">Imad eddine Djehlane · data: KuaiRec (Gao et al., 2022)</span>
  </header>

  <h1>what is your scrolling <span>made of?</span></h1>
  <p class="lead">Food comes with a label. Short video doesn't. This prototype turns real Kuaishou viewing logs into a
    one-glance label: how much, what kind, how much of it is replays and late-night viewing, and a short summary written by an AI model.</p>

  <div class="picker" id="picker"></div>
  <div class="note" id="note"></div>

  <div class="main">
    <div class="bottle">
      <div class="mono" style="margin-bottom:10px">Your viewing, bottled (share of watch time)</div>
      <div id="bottle"></div>
      <div class="legend" id="legend"></div>
    </div>
    <div id="label"></div>
    <div class="ai">
      <div class="mono" style="margin-bottom:10px">AI summary</div>
      <div class="card">
        <div class="q" id="summary"></div>
        <div class="mono src" id="src"></div>
      </div>
      <div class="why">
        <b style="color:var(--fg)">How the summary is made</b>
        <ul>
          <li>The model sees only the numbers on the label, not your watch history.</li>
          <li>It must use only those numbers, stay neutral and give one optional suggestion.</li>
          <li>No diagnosis, no moralising. The label informs, you decide.</li>
        </ul>
      </div>
    </div>
  </div>

  <h2>three moments</h2>
  <p class="sub">A label only helps if it shows up at the right time. These mock-ups use the selected user's numbers.</p>
  <div class="moments">
    <div class="moment">
      <div class="mono">01 · while scrolling</div><h4>a clock in the casino</h4>
      <div class="phone"><div class="screen"><div class="play"></div><div class="chip" id="m1"></div></div></div>
      <p class="cap">A small, always-visible counter. It doesn't block anything: it just puts the clock back.</p>
    </div>
    <div class="moment">
      <div class="mono">02 · at your limit</div><h4>a pause, not a wall</h4>
      <div class="phone"><div class="screen"><div class="play"></div><div class="sheet" id="m2"></div></div></div>
      <p class="cap">The user sets the limit. When it's reached, the feed pauses and shows what the sitting was made of. One tap continues.</p>
    </div>
    <div class="moment">
      <div class="mono">03 · next morning</div><h4>the daily label</h4>
      <div class="phone"><div class="screen"><div class="mini" id="m3"></div></div></div>
      <p class="cap">The full label and the AI summary, once a day, when the user isn't in the feed.</p>
    </div>
  </div>

  <h2>who can build this?</h2>
  <p class="sub">The platform could build all three moments tomorrow: it already has every number. An outside app (a "third party") can only do part of it.</p>
  <table class="who">
    <tr><th></th><th>The platform (TikTok, Kuaishou)</th><th>A third party (browser plug-in, independent app)</th></tr>
    <tr><td>01 · while scrolling</td><td class="yes">Yes, it owns the screen</td><td class="no">Only on the web version, via a browser plug-in. Inside the mobile apps: no, outside apps can't read another app's screen.</td></tr>
    <tr><td>02 · at your limit</td><td class="yes">Yes</td><td class="no">Partly: phone settings (Screen Time, Digital Wellbeing) can limit total app time, but can't tell what the time was made of.</td></tr>
    <tr><td>03 · next morning</td><td class="yes">Yes</td><td class="yes">Yes, but delayed: the user can download their own watch history (GDPR right of access, Art. 15, and data portability, Art. 20). The export has no categories, so an AI model would have to tag the videos from their titles.</td></tr>
    <tr><td>Change the feed itself</td><td class="yes">Yes</td><td class="no">No. A third party can inform the user, not change what the recommender optimises.</td></tr>
  </table>

  <div class="notes" id="notes"></div>
</div>

<script>
const DATA = /*__DATA__*/null;
const COLORS = ["#e85d26", "#1aa3ad", "#b8860b", "#8b72d6", "#d95f8f", "#5c5c54"]; // validated for CVD on #111
const P = DATA.pop, N = DATA.n_users;
const pct = x => Math.round(x * 100) + "%";
const fmt = x => (x >= 10 ? Math.round(x) : x.toFixed(1));
const tip = document.getElementById("tip");
const esc = s => String(s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));

function cmpText(p) {
  return p >= 0.5 ? "higher than " + Math.round(p * 100) + "% of users" : "lower than " + Math.round((1 - p) * 100) + "% of users";
}
function bar(p) { return `<span class="bar ${p >= 0.8 ? "hi" : ""}"><i style="left:${(p * 100).toFixed(1)}%"></i></span>`; }

function bottle(u) {
  const top = 132, bottom = 340, H = bottom - top;
  let y = bottom, rects = "";
  u.ingredients.forEach((g, i) => {
    const h = g.share * H;
    y -= h;
    rects += `<rect class="layer" data-t="${esc(g.name)}: ${pct(g.share)} of watch time" x="0" y="${y.toFixed(1)}" width="200" height="${h.toFixed(1)}" fill="${COLORS[i]}" stroke="#111" stroke-width="2"/>`;
  });
  const path = "M80 20 H120 V70 Q120 90 150 105 Q170 115 170 140 V320 Q170 340 150 340 H50 Q30 340 30 320 V140 Q30 115 50 105 Q80 90 80 70 Z";
  document.getElementById("bottle").innerHTML =
    `<svg viewBox="0 0 200 360" role="img" aria-label="Bottle filled with viewing categories"><defs><clipPath id="cb"><path d="${path}"/></clipPath></defs>
     <g clip-path="url(#cb)">${rects}</g>
     <path d="${path}" fill="none" stroke="#f0ece5" stroke-width="3"/>
     <rect x="76" y="10" width="48" height="12" rx="2" fill="#f0ece5"/></svg>`;
  document.getElementById("legend").innerHTML = u.ingredients.map((g, i) =>
    `<div><i style="background:${COLORS[i]}"></i><span>${esc(g.name)}</span><b>${pct(g.share)}</b></div>`).join("");
  document.querySelectorAll(".layer").forEach(el => {
    el.addEventListener("mousemove", e => { tip.style.display = "block"; tip.textContent = el.dataset.t; tip.style.left = e.clientX + 12 + "px"; tip.style.top = e.clientY + 12 + "px"; });
    el.addEventListener("mouseleave", () => tip.style.display = "none");
  });
}

function label(u) {
  const ing = u.ingredients.map((g, i) => `<div class="row ing ${i ? "" : "first"}"><span>${esc(g.name)}</span><span>${pct(g.share)}</span></div>`).join("");
  const line = (name, val, p, med) => `<div class="row"><span><b>${name}</b></span><span><b>${val}</b>${bar(p)}</span>
      <span class="cmp">${cmpText(p)} · median user: ${med}</span></div>`;
  document.getElementById("label").innerHTML = `<div class="label">
    <h3>Video Nutrition Facts</h3>
    <div class="srv">Serving size: one average day<br>Based on ${u.days} days of viewing · user ${u.user_id}</div>
    <div class="row big"><span>Videos per day</span><span class="v">${fmt(u.videos_per_day)}</span>
      <span class="cmp">${cmpText(u.pct.videos_per_day)} · median user: ${fmt(P.videos_per_day.p50)}</span></div>
    <div class="row"><span>Logged watch time*</span><span>${fmt(u.minutes_per_day)} min</span></div>
    <div class="head thick">% of your watch time</div>
    ${ing}
    <div class="row thick"><span><b>Variety</b>: 80% of your time is in</span><span><b>${u.cats_for_80} of 39</b> categories</span>
      <span class="cmp">median user: ${fmt(P.cats_for_80.p50)} categories</span></div>
    ${line("Replays", pct(u.replay_share) + " of time", u.pct.replay_share, pct(P.replay_share.p50))}
    ${line("Night viewing (00:00–05:59)", pct(u.night_share) + " of time", u.pct.night_share, pct(P.night_share.p50))}
    ${line("Quick swipes (&lt;¼ watched)", pct(u.skip_share) + " of videos", u.pct.skip_share, pct(P.skip_share.p50))}
    ${line("Longest sitting", u.longest_sitting + " videos", u.pct.longest_sitting, fmt(P.longest_sitting.p50) + " videos")}
    <div class="foot">Compared with ${N.toLocaleString("en")} KuaiRec users with a complete viewing history. There is no official healthy amount of short video,
      so the label shows how you compare with others, not a target. The marker shows where you sit; orange = top 20%.
      Replays = seconds watched beyond one full play. Sitting = videos with no 30-minute break.<br>
      * KuaiRec logs only part of what people watch, so minutes are understated. Video counts and shares are the reliable part.</div>
  </div>`;
}

function moments(u) {
  const perSit = Math.max(1, Math.round(u.videos_per_day / Math.max(u.sittings_per_day, 1)));
  const top = u.ingredients[0];
  document.getElementById("m1").innerHTML = `<b>${perSit}</b> videos this sitting · <b>${fmt(u.videos_per_day)}</b> a day on average<br>mostly ${esc(top.name)} (${pct(top.share)})`;
  const stack = u.ingredients.map((g, i) => `<span title="${esc(g.name)}" style="flex:${g.share};background:${COLORS[i]}"></span>`).join("");
  document.getElementById("m2").innerHTML = `<h5>You reached your limit</h5>You set ${Math.max(10, Math.round(u.videos_per_day / 10) * 10)} videos a day. Today's mix:
    <div class="stack">${stack}</div>${esc(top.name)} ${pct(top.share)} · replays ${pct(u.replay_share)}
    <div class="btns"><div class="p">Stop here</div><div>10 more</div></div>`;
  document.getElementById("m3").innerHTML = `<h5>Yesterday, bottled</h5>
    <b>${fmt(u.videos_per_day)} videos</b> · ${u.cats_for_80} categories made 80%<br>
    replays ${pct(u.replay_share)} · night ${pct(u.night_share)}<div class="stack">${stack}</div>
    <p>${esc(u.summary)}</p>`;
}

function show(uid) {
  const u = DATA.users.find(x => x.user_id === uid);
  bottle(u); label(u); moments(u);
  document.getElementById("summary").textContent = u.summary;
  document.getElementById("src").textContent = "Source: " + u.summary_source;
  const note = document.getElementById("note");
  note.style.display = u.shared_sample ? "block" : "none";
  note.innerHTML = `User ${u.user_id} is in the 100-user sample shared with Roles 2 and 3. Careful: KuaiRec removed these users' test-set videos
    from their viewing history, so their label is incomplete (they look like light viewers: median ${fmt(DATA.shared_median)} videos a day vs ${fmt(P.videos_per_day.p50)} for users with a complete history). Use it to connect with the AI agent demo, not to judge the user.`;
  document.querySelectorAll(".picker button").forEach(b => b.classList.toggle("on", +b.dataset.u === uid));
  document.getElementById("sel").value = u.persona ? "" : uid;
}

const demos = DATA.users.filter(u => u.persona);
const shared = DATA.users.filter(u => u.shared_sample && !u.persona);
document.getElementById("picker").innerHTML =
  demos.map(u => `<button data-u="${u.user_id}">${esc(u.persona)}</button>`).join("") +
  `<select id="sel"><option value="">Roles 2–3 shared users…</option>${shared.map(u => `<option value="${u.user_id}">user ${u.user_id}</option>`).join("")}</select>`;
document.querySelectorAll(".picker button").forEach(b => b.addEventListener("click", () => show(+b.dataset.u)));
document.getElementById("sel").addEventListener("change", e => e.target.value && show(+e.target.value));

document.getElementById("notes").innerHTML = `
  <p><b>Data.</b> KuaiRec big matrix: ${N.toLocaleString("en")} users with a complete history (the 1,411 test-set users are left out of all comparisons, because KuaiRec moved part of their viewing into the test set), ${DATA.n_days} logging days between July and September 2020 on Kuaishou.
  One category per video (the platform's own first-level category, translated). Gao et al. (2022), KuaiRec: A Fully-Observed Dataset and
  Insights for Evaluating Recommender Systems, CIKM '22.</p>
  <p><b>What the data can't show.</b> KuaiRec contains 10,728 videos, not the whole app. Logged play time is only about ${pct(DATA.coverage)} of the
  time between the first and last logged video of a sitting, so real watch time is much higher than the minutes shown. Counts, shares and
  comparisons between users are what this prototype relies on.</p>
  <p><b>Night viewing</b> uses the clock stored in the data. Its daily pattern peaks in the late morning, which is unusual for a video app,
  so treat the night line as illustrative.</p>
  <p><b>Demo users</b> were picked by a fixed rule, not by hand: among complete users with a typical amount of viewing (40th–60th percentile of videos
  per day), the one with the narrowest and the broadest mix, plus the user at the 95th percentile of videos per day.</p>`;
DATA.shared_median = DATA.shared_median || P.videos_per_day.p50;
show(demos[0].user_id);
</script>
</body>
</html>
'''
print(len(TEMPLATE), 'characters of HTML template loaded')

In [ ]:
cards = []
for uid in list(DEMO) + [u for u in shared if u not in DEMO]:
    d = label_dict(uid)
    ai = summaries.get(str(uid))
    d["summary"] = ai["text"] if ai else rule_summary(d)
    d["summary_source"] = ai["source"] if ai else "Rule-based text (no AI key was used for this user)"
    cards.append(d)
payload = {"users": cards, "pop": {m: pop[m] for m in METRICS}, "n_users": pop["n_users"],
           "coverage": pop["coverage"], "n_days": pop["n_days_in_data"],
           "shared_median": pop["shared_median_videos_per_day"]}
json.dump(payload, open(OUT / "labels_shared_users.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)

template = TEMPLATE   # defined in the last cell of Step 8's setup (pasted HTML)
page = template.replace("/*__DATA__*/null", json.dumps(payload, ensure_ascii=False))
open(OUT / "label_prototype.html", "w", encoding="utf-8").write(page)
print("Saved", OUT / "label_prototype.html", "- download it from the file panel and open it in a browser.")